<a href="https://colab.research.google.com/github/5atish/TrainingCourse/blob/main/Hugging%20Face%20Agents%20Course%20/Unit%202.%20Frameworks%20for%20AI%20Agents/Unit%202.1%20The%20smolagents%20framework/Multi_Agent_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install 'smolagents[litellm]' plotly geopandas shapely kaleido -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.7 MB/s eta 0:00:00


In [2]:
import math
from typing import Optional, Tuple

from smolagents import tool


@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0

    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)


print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))

22.82


In [3]:
import os
from PIL import Image
from smolagents import CodeAgent, GoogleSearchTool, InferenceClientModel, VisitWebpageTool

model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct", provider="together")

In [4]:
task = """Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.
Also give me some supercar factories with the same cargo plane transfer time."""

In [19]:
import os
print(bool(os.environ.get("SERPER_API_KEY")))

True


In [15]:
from google.colab import userdata
import os

os.environ["SERPER_API_KEY"] = userdata.get('SERPAPI_API_KEY')
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [16]:
model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-72B-Instruct",
    provider="auto",
    token=os.environ["HF_TOKEN"]
)

In [17]:
agent = CodeAgent(
    model=model,
    tools=[GoogleSearchTool("serper"), VisitWebpageTool(), calculate_cargo_travel_time],
    additional_authorized_imports=["pandas"],
    max_steps=20,
)

In [18]:
result = agent.run(task)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  batman_filming_locations = web_search(query="all Batman filming locations")                                      
  print(batman_filming_locations)                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'batman_filming_locations = web_search(query="all Batman filming locations")' due to:
ValueError: {'message': 'Unauthorized.', 'statusCode': 403}

[Step 1: Duration 1.36 seconds| Input tokens: 2,323 | Output tokens: 45]

...

[Step 5: Duration 30.39 seconds| Input tokens: 13,874 | Output tokens: 1,851]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 6 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(df_combined)                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer:                     name  travel_time
0            Chicago, IL         2.68
1             London, UK         9.17
2  Vancouver, BC, Canada         6.73
0                Ferrari        10.73
1            Lamborghini        10.76
2                McLaren         8.99

[Step 6: Duration 1.15 seconds| Input tokens: 18,504 | Output tokens: 1,893]

In [20]:
result

,name,travel_time
0,"Chicago, IL",2.68
1,"London, UK",9.17
2,"Vancouver, BC, Canada",6.73
0,Ferrari,10.73
1,Lamborghini,10.76
2,McLaren,8.99


In [21]:
agent.planning_interval = 4

detailed_report = agent.run(f"""
You're an expert analyst. You make comprehensive reports after visiting many websites.
Don't hesitate to search for many queries at once in a for loop.
For each data point that you find, visit the source url to confirm numbers.

{task}
""")

print(detailed_report)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You're an expert analyst. You make comprehensive reports after visiting many websites.                          │
│ Don't hesitate to search for many queries at once in a for loop.                                                │
│ For each data point that you find, visit the source url to confirm numbers.                                     │
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
## 1. Facts Survey

### 1.1. Facts given in the task
- Our current location: Gotham, 40.7128° N, 74.0060° W
- We need to find all Batman filming locations in the world.
- We need to calculate the travel time for cargo planes from each filming location to Gotham.
- We need to find some supercar factories and calculate their travel time to Gotham using cargo planes.

### 1.2. Facts to look up
- List of all Batman filming locations worldwide.
- Coordinates (latitude, longitude) for each Batman filming location.
- List of supercar factories and their coordinates (latitude, longitude).
- Cruising speed of cargo planes (default is 750 km/h).

### 1.3. Facts to derive
- Travel time from each Batman filming location to Gotham using the `calculate_cargo_travel_time` function.
- Travel time from each supercar factory to Gotham using the `calculate_cargo_travel_time` function.
- Compilation of all the data into a pandas dataframe.

## 2. Plan

1. Perform a web search to find a comprehensive list of all Batman filming locations worldwide.
2. For each Batman filming location, find its coordinates (latitude, longitude).
3. Use the `calculate_cargo_travel_time` function to calculate the travel time from each filming location to 
Gotham.
4. Perform a web search to find a list of supercar factories and their coordinates.
5. Use the `calculate_cargo_travel_time` function to calculate the travel time from each supercar factory to 
Gotham.
6. Compile all the data (filming locations, supercar factories, coordinates, and travel times) into a pandas 
dataframe.
7. Return the compiled dataframe using the `final_answer` function.

```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import pandas as pd                                                                                              
                                                                                                                   
  # Define the query for finding Batman filming locations                                                          
  query = "Batman filming locations worldwide"                                                                     
                                                                                                                   
  # Perform the web search to find the Batman filming locations                                                    
  pages = web_search(query=query)                                                                                  
  print(pages)                                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'pages = web_search(query=query)' due to: ValueError: {'message': 'Unauthorized.', 
'statusCode': 403}

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Manually gathered Batman filming locations and their coordinates                                               
  filming_locations = [                                                                                            
      {"location": "Chicago, Illinois, USA", "coords": (41.8781, -87.6298)},                                       
      {"location": "Detroit, Michigan, USA", "coords": (42.3314, -83.0458)},                                       
      {"location": "London, England, UK", "coords": (51.5074, -0.1278)},                                           
      {"location": "Liverpool, England, UK", "coords": (53.4106, -2.9779)},                                        
      {"location": "Vancouver, British Columbia, Canada", "coords": (49.2827, -123.1207)},                         
      {"location": "Berlin, Germany", "coords": (52.5200, 13.4050)},                                               
      {"location": "Zug Island, Detroit, USA", "coords": (42.3221, -83.0125)},                                     
      {"location": "Ludlow, California, USA", "coords": (35.1947, -116.1784)},                                     
      {"location": "Istanbul, Turkey", "coords": (41.0082, 28.9784)},                                              
      {"location": "Shanghai, China", "coords": (31.2304, 121.4737)},                                              
  ]                                                                                                                
                                                                                                                   
  # Coordinates of Gotham (New York City)                                                                          
  gotham_coords = (40.7128, -74.0060)                                                                              
                                                                                                                   
  # Initialize lists to store data                                                                                 
  data = []                                                                                                        
                                                                                                                   
  # Calculate travel times for each filming location                                                               
  for location in filming_locations:                                                                               
      origin_coords = location["coords"]                                                                           
      travel_time = calculate_cargo_travel_time(origin_coords, gotham_coords)                                      
      data.append({"Location": location["location"], "Coordinates": [38;2;248;248;24

Execution logs:
                               Location           Coordinates  \
0                Chicago, Illinois, USA   (41.8781, -87.6298)   
1                Detroit, Michigan, USA   (42.3314, -83.0458)   
2                   London, England, UK    (51.5074, -0.1278)   
3                Liverpool, England, UK    (53.4106, -2.9779)   
4   Vancouver, British Columbia, Canada  (49.2827, -123.1207)   
5                       Berlin, Germany       (52.52, 13.405)   
6              Zug Island, Detroit, USA   (42.3221, -83.0125)   
7               Ludlow, California, USA  (35.1947, -116.1784)   
8                      Istanbul, Turkey    (41.0082, 28.9784)   
9                       Shanghai, China   (31.2304, 121.4737)   
10                     Maranello, Italy    (44.3917, 10.7852)   
11          Sant'Agata Bolognese, Italy    (44.4379, 11.2595)   
12                           Gaydon, UK    (52.2803, -1.6372)   
13                    Wolfburg, Germany     (52.078, 10.6698)   

    Travel Time (hours)  
0                  2.68  
1                  2.13  
2                  9.17  
3                  8.81  
4                  6.73  
5                 10.36  
6                  2.13  
7                  6.44  
8                 12.84  
9                 18.39  
10                10.73  
11                10.77  
12                 8.99  
13                10.15

Final answer:                                Location           Coordinates  \
0                Chicago, Illinois, USA   (41.8781, -87.6298)   
1                Detroit, Michigan, USA   (42.3314, -83.0458)   
2                   London, England, UK    (51.5074, -0.1278)   
3                Liverpool, England, UK    (53.4106, -2.9779)   
4   Vancouver, British Columbia, Canada  (49.2827, -123.1207)   
5                       Berlin, Germany       (52.52, 13.405)   
6              Zug Island, Detroit, USA   (42.3221, -83.0125)   
7               Ludlow, California, USA  (35.1947, -116.1784)   
8                      Istanbul, Turkey    (41.0082, 28.9784)   
9                       Shanghai, China   (31.2304, 121.4737)   
10                     Maranello, Italy    (44.3917, 10.7852)   
11          Sant'Agata Bolognese, Italy    (44.4379, 11.2595)   
12                           Gaydon, UK    (52.2803, -1.6372)   
13                    Wolfburg, Germany     (52.078, 10.6698)   

    Travel Time (hours)  
0                  2.68  
1                  2.13  
2                  9.17  
3                  8.81  
4                  6.73  
5                 10.36  
6                  2.13  
7                  6.44  
8                 12.84  
9                 18.39  
10                10.73  
11                10.77  
12                 8.99  
13                10.15  

[Step 4: Duration 57.51 seconds| Input tokens: 13,090 | Output tokens: 3,553]

                               Location           Coordinates  \
0                Chicago, Illinois, USA   (41.8781, -87.6298)   
1                Detroit, Michigan, USA   (42.3314, -83.0458)   
2                   London, England, UK    (51.5074, -0.1278)   
3                Liverpool, England, UK    (53.4106, -2.9779)   
4   Vancouver, British Columbia, Canada  (49.2827, -123.1207)   
5                       Berlin, Germany       (52.52, 13.405)   
6              Zug Island, Detroit, USA   (42.3221, -83.0125)   
7               Ludlow, California, USA  (35.1947, -116.1784)   
8                      Istanbul, Turkey    (41.0082, 28.9784)   
9                       Shanghai, China   (31.2304, 121.4737)   
10                     Maranello, Italy    (44.3917, 10.7852)   
11          Sant'Agata Bolognese, Italy    (44.4379, 11.2595)   
12                           Gaydon, UK    (52.2803, -1.6372)   
13                    Wolfburg, Germany     (52.078, 10.6698)   

    Travel Time (hours) 

In [22]:
detailed_report

,Location,Coordinates,Travel Time (hours)
0,"Chicago, Illinois, USA","(41.8781, -87.6298)",2.68
1,"Detroit, Michigan, USA","(42.3314, -83.0458)",2.13
2,"London, England, UK","(51.5074, -0.1278)",9.17
3,"Liverpool, England, UK","(53.4106, -2.9779)",8.81
4,"Vancouver, British Columbia, Canada","(49.2827, -123.1207)",6.73
5,"Berlin, Germany","(52.52, 13.405)",10.36
6,"Zug Island, Detroit, USA","(42.3221, -83.0125)",2.13
7,"Ludlow, California, USA","(35.1947, -116.1784)",6.44
8,"Istanbul, Turkey","(41.0082, 28.9784)",12.84
9,"Shanghai, China","(31.2304, 121.4737)",18.39


In [23]:
model = InferenceClientModel(
    "Qwen/Qwen2.5-Coder-32B-Instruct", provider="together", max_tokens=8096
)

web_agent = CodeAgent(
    model=model,
    tools=[
        GoogleSearchTool(provider="serper"),
        VisitWebpageTool(),
        calculate_cargo_travel_time,
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=0,
    max_steps=10,
)

In [38]:
from smolagents.utils import encode_image_base64, make_image_url
from smolagents import OpenAIServerModel


def check_reasoning_and_plot(final_answer, agent_memory):
    multimodal_model = OpenAIServerModel("gpt-4o", max_tokens=8096)
    filepath = "saved_map.png"
    assert os.path.exists(filepath), "Make sure to save the plot under saved_map.png!"
    image = Image.open(filepath)
    prompt = (
        f"Here is a user-given task and the agent steps: {agent_memory.get_succinct_steps()}. Now here is the plot that was made."
        "Please check that the reasoning process and plot are correct: do they correctly answer the given task?"
        "First list reasons why yes/no, then write your final decision: PASS in caps lock if it is satisfactory, FAIL if it is not."
        "Don't be harsh: if the plot mostly solves the task, it should pass."
        "To pass, a plot should be made using px.scatter_map and not any other method (scatter_map looks nicer)."
    )
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
                {
                    "type": "image_url",
                    "image_url": {"url": make_image_url(encode_image_base64(image))},
                },
            ],
        }
    ]
    output = multimodal_model(messages).content
    print("Feedback: ", output)
    if "FAIL" in output:
        raise Exception(output)
    return True


manager_agent = CodeAgent(
    model=InferenceClientModel("meta-llama/Llama-3.3-70B-Instruct", provider="together", max_tokens=8096),
    tools=[calculate_cargo_travel_time],
    managed_agents=[web_agent],
    additional_authorized_imports=[
        "geopandas",
        "plotly",
        "shapely",
        "json",
        "pandas",
        "numpy",
    ],
    planning_interval=5,
    verbosity_level=2,
    final_answer_checks=[check_reasoning_and_plot],
    max_steps=15,
)

In [39]:
manager_agent.visualize()

CodeAgent | meta-llama/Llama-3.3-70B-Instruct
├── ✅ Authorized imports: ['geopandas', 'plotly', 'shapely', 'json', 'pandas', 'numpy']
├── 🛠️ Tools:
│   ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
│   ┃ Name                        ┃ Description                           ┃ Arguments                             ┃
│   ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │ calculate_cargo_travel_time │ Calculate the travel time for a cargo │ origin_coords (`array`): Tuple of     │
│   │                             │ plane between two points on Earth     │ (latitude, longitude) for the         │
│   │                             │ using great-circle distance.          │ starting point                        │
│   │                             │                                       │ destination_coords (`array`): Tuple   │
│   │                             │                                       │ of (latitude, longitude) for the      │
│   │                             │                                       │ destination                           │
│   │                             │                                       │ cruising_speed_kmh (`number`):        │
│   │                             │                                       │ Optional cruising speed in km/h       │
│   │                             │                                       │ (defaults to 750 km/h for typical     │
│   │                             │                                       │ cargo planes)                         │
│   │ final_answer                │ Provides a final answer to the given  │ answer (`any`): The final answer to   │
│   │                             │ problem.                              │ the problem                           │
│   └─────────────────────────────┴───────────────────────────────────────┴───────────────────────────────────────┘
└── 🤖 Managed agents:
    └── web_agent | CodeAgent | Qwen/Qwen2.5-Coder-32B-Instruct
        ├── ✅ Authorized imports: []
        ├── 📝 Description: Browses the web to find information
        └── 🛠️ Tools:
            ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
            ┃ Name                        ┃ Description                       ┃ Arguments                         ┃
            ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
            │ web_search                  │ Performs a google web search for  │ query (`string`): The search      │
            │                             │ your query then returns a string  │ query to perform.                 │
            │                             │ of the top search results.        │ filter_year (`integer`):          │
            │                             │                                   │ Optionally restrict results to a  │
            │                             │                                   │ certain year                      │
            │ visit_webpage               │ Visits a webpage at the given url │ url (`string`): The url of the    │
            │                             │ and reads its content as a        │ webpage to visit.                 │
            │                             │ markdown string. Use this to      │                                   │
            │                             │ browse webpages.                  │                                   │
            │ calculate_cargo_travel_time │ Calculate the travel time for a   │ origin_coords (`array`): Tuple of │
            │                             │ cargo plane between two points on │ (latitude, longitude) for the     │
            │                             │ Earth using great-circle          │ starting point                    │
            │                             │ dist

In [33]:
# model = InferenceClientModel(
#     model_id="deepseek-ai/DeepSeek-R1",
#     provider="auto",
#     token=os.environ["HF_TOKEN"]
# )

In [40]:
manager_agent.run("""
Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W).
Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in total.
Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time, and save it to saved_map.png!

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
     color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)
fig.show()
fig.write_image("saved_image.png")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
""")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W).                                                                             │
│ Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in     │
│ total.                                                                                                          │
│ Represent this as spatial map of the world, with the locations represented as scatter points with a color that  │
│ depends on the travel time, and save it to saved_map.png!                                                       │
│                                                                                                                 │
│ Here's an example of how to plot and return a map:                                                              │
│ import plotly.express as px                                                                                     │
│ df = px.data.carshare()                                                                                         │
│ fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,      │
│      color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)                                    │
│ fig.show()                                                                                                      │
│ fig.write_image("saved_image.png")                                                                              │
│ final_answer(fig)                                                                                               │
│                                                                                                                 │
│ Never try to process strings using code: when you have a string to read, just print it and you'll see it.       │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.3-70B-Instruct ──────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
## 1. Facts survey
### 1.1. Facts given in the task
- Current location: Gotham, 40.7128° N, 74.0060° W
- Task: Find Batman filming locations and supercar factories with similar cargo plane transfer times
- Desired output: A spatial map with locations as scatter points colored by travel time

### 1.2. Facts to look up
- Batman filming locations around the world
  - Source: Web search using `web_agent`
- Supercar factories around the world
  - Source: Web search using `web_agent`
- Cruising speed of a typical cargo plane (if not using the default 750 km/h)
  - Source: Web search or aviation databases

### 1.3. Facts to derive
- Travel time from Gotham to each Batman filming location
  - Derived using: `calculate_cargo_travel_time` function with origin as Gotham and destinations as filming 
locations
- Travel time from Gotham to each supercar factory
  - Derived using: `calculate_cargo_travel_time` function with origin as Gotham and destinations as supercar 
factories
- Locations with similar cargo plane transfer times
  - Derived by: Comparing travel times calculated for filming locations and supercar factories

## 2. Plan
1. Use `web_agent` to find all Batman filming locations worldwide.
2. Use `web_agent` to find supercar factories worldwide.
3. For each filming location and supercar factory, use `calculate_cargo_travel_time` to calculate the travel time 
from Gotham.
4. Filter locations to include at least 6 points with similar cargo plane transfer times.
5. Create a dataframe with the locations, their respective travel times, and other necessary details for plotting.
6. Use Plotly Express to create a scatter map with locations colored by travel time.
7. Save the map as "saved_map.png".
8. Provide the final answer using `final_answer` with the generated map.

```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I need to find all Batman filming locations worldwide and supercar factories worldwide.                   
<code>                                                                                                             
batman_filming_locations = web_agent(task="Find all Batman filming locations worldwide", additional_args={})       
print(batman_filming_locations)                                                                                    
                                                                                                                   
supercar_factories = web_agent(task="Find supercar factories worldwide", additional_args={})                       
print(supercar_factories)                                                                                          
</code>                                                                                                            
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  batman_filming_locations = web_agent(task="Find all Batman filming locations worldwide", additional_args={})     
  print(batman_filming_locations)                                                                                  
                                                                                                                   
  supercar_factories = web_agent(task="Find supercar factories worldwide", additional_args={})                     
  print(supercar_factories)                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

[Step 2: Duration 2.13 seconds| Input tokens: 6,223 | Output tokens: 211]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/together/v1/chat/completions' (Request 
ID: Root=1-6a93405d-5948aa522e10b5cf0e9b48ff;57790170-43b0-4e34-a551-91a28f08b67d)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 3: Duration 0.04 seconds]

AgentGenerationError: Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/together/v1/chat/completions' (Request ID: Root=1-6a93405d-5948aa522e10b5cf0e9b48ff;57790170-43b0-4e34-a551-91a28f08b67d)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

In [ ]:
manager_agent.python_executor.state["fig"]